# Task 2 — Street permits

Run cells from top to bottom in a fresh Python/Colab session. Dependencies: pandas >= 2, numpy, requests and plotly >= 5.5.

**Inputs:** `task2_street_raw_backup.zip` and `task3_results.zip` (or corresponding CSV files).

Saved raw snapshots are the reproducible inputs. Live API extraction code is included separately in the bundle for provenance and is not run by these notebooks. Expected totals describe the supplied snapshot; a different snapshot needs new validation and updated findings.

**Scope:** all five boroughs, issuance dates in 2023–2025. Building source: https://data.cityofnewyork.us/dataset/rbx6-tga4 . Street source: https://data.cityofnewyork.us/d/tqtj-sjs8 .


In [ ]:
# Load existing CSVs or ZIP backups locally; upload only when an input is missing.
# Only the requested CSV member is read. ZIP contents are not extracted to disk.
# Raw snapshots are reused; this workflow does not redownload live data.
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
from IPython.display import display
try:
    import google.colab
    BASE = Path('/content')
except ImportError:
    BASE = Path.cwd()
DATA_DIR = BASE / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
def load_saved_csv(filename, **kwargs):
    roots = [DATA_DIR, BASE]
    def locate():
        for folder in roots:
            direct = folder / filename
            if direct.exists():
                return pd.read_csv(direct, **kwargs)
        archives = sorted({p for folder in roots for p in folder.glob('*.zip')})
        for path in archives:
            try:
                with ZipFile(path) as archive:
                    members = [n for n in archive.namelist()
                               if Path(n).name == filename]
                    if len(members) == 1:
                        with archive.open(members[0]) as handle:
                            return pd.read_csv(handle, **kwargs)
            except zipfile.BadZipFile:
                continue
        return None
    import zipfile
    result = locate()
    if result is None:
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError(f'Place {filename} or its ZIP in {DATA_DIR}') from exc
        print(f'Upload {filename} or the ZIP containing it.')
        files.upload()
        result = locate()
    if result is None:
        raise FileNotFoundError(f'Uploaded files did not contain {filename}.')
    return result

In [ ]:
# Restore the raw street snapshot while retaining text identifiers.
import numpy as np
import plotly.express as px
STREET_OUTPUT_DIR = BASE / 'task2_outputs'
STREET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_street = load_saved_csv('street_permits_raw_2023_2025.csv', dtype='string')
assert df_street.shape == (1965229, 20), 'Unexpected street snapshot.'

In [ ]:
# Inspect missing values, categories and permit identifiers before cleaning.
print(f"Rows: {len(df_street):,}")
print(f"Columns: {len(df_street.columns)}")
print(
    "Memory used:",
    f"{df_street.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
)

quality = pd.DataFrame({
    "missing_count": df_street.isna().sum(),
    "missing_percent": (
        df_street.isna().mean() * 100
    ).round(2)
}).sort_values("missing_count", ascending=False)

print("\nMISSING VALUES")
display(quality)

print("\nPERMIT IDENTIFIER CHECK")
print("Missing permit numbers:", df_street["permitnumber"].isna().sum())
print(
    "Repeated permit numbers:",
    df_street["permitnumber"].duplicated().sum()
)

for column in [
    "boroughname",
    "permitstatusshortdesc",
    "applicationtypeshortdesc",
    "permitseriesshortdesc"
]:
    print(f"\n{column.upper()}")
    display(
        df_street[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="record_count")
    )

quality.to_csv(
    STREET_OUTPUT_DIR / "street_initial_missing_values.csv"
)

print("\nReal-data inspection complete.")

In [8]:
# This cell standardizes text, converts dates and numeric measurements,
# and gives core fields clear names for integration.
# Missing measurements remain unknown rather than being replaced with zero.
# Identifiers remain text, and conversion failures are reported.
# We modify the working table to avoid duplicating a large dataset in memory.

for column in df_street.select_dtypes(include=["string", "object"]).columns:
    df_street[column] = (
        df_street[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

conversion_report = []

date_columns = [
    "permitissuedate",
    "issuedworkstartdate",
    "issuedworkenddate"
]

for column in date_columns:
    original = df_street[column]
    converted = pd.to_datetime(
        original,
        format="ISO8601",
        errors="coerce"
    )

    conversion_report.append({
        "column": column,
        "conversion_failures": int(
            (original.notna() & converted.isna()).sum()
        )
    })

    df_street[column] = converted

numeric_columns = [
    "permitlinearfeet",
    "permittotalsqfeet",
    "permitestimatednumberofcuts"
]

for column in numeric_columns:
    original = df_street[column]
    converted = pd.to_numeric(original, errors="coerce")

    conversion_report.append({
        "column": column,
        "conversion_failures": int(
            (original.notna() & converted.isna()).sum()
        )
    })

    df_street[column] = converted

df_street.rename(columns={
    "boroughname": "borough",
    "permitissuedate": "issued_date",
    "permitstatusshortdesc": "permit_status",
    "permitseriesshortdesc": "permit_series",
    "permittypedesc": "permit_category",
    "applicationtypeshortdesc": "application_type"
}, inplace=True)

df_street["borough"] = df_street["borough"].str.upper()

# Store repeated labels efficiently without changing their meaning.
for column in [
    "borough", "permit_status", "permit_series",
    "permit_category", "application_type"
]:
    df_street[column] = df_street[column].astype("category")

valid_boroughs = {
    "BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND"
}

print("CONVERSION CHECKS")
display(pd.DataFrame(conversion_report))

print(
    "\nMissing or unexpected boroughs:",
    int((~df_street["borough"].isin(valid_boroughs)).sum())
)

print(
    "Missing permit numbers after text cleaning:",
    int(df_street["permitnumber"].isna().sum())
)

print(
    "Repeated permit numbers after text cleaning:",
    int(df_street["permitnumber"].duplicated().sum())
)

valid_scope = (
    df_street["issued_date"].ge(pd.Timestamp("2023-01-01"))
    & df_street["issued_date"].lt(pd.Timestamp("2026-01-01"))
)

print("Missing or out-of-scope issuance dates:", int((~valid_scope).sum()))
print(
    "Issuance date range:",
    df_street["issued_date"].min(),
    "to",
    df_street["issued_date"].max()
)

print(f"Rows retained: {len(df_street):,}")
print(
    "Memory used:",
    f"{df_street.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
)

CONVERSION CHECKS


,column,conversion_failures
0,permitissuedate,0
1,issuedworkstartdate,0
2,issuedworkenddate,0
3,permitlinearfeet,0
4,permittotalsqfeet,0
5,permitestimatednumberofcuts,0



Missing or unexpected boroughs: 0
Missing permit numbers after text cleaning: 0
Repeated permit numbers after text cleaning: 0
Missing or out-of-scope issuance dates: 0
Issuance date range: 2023-01-01 01:47:44 to 2025-12-31 23:14:15
Rows retained: 1,965,229
Memory used: 1118.4 MB


In [9]:
# This cell flags negative measurements, unusually high values and work-date issues.
# IQR thresholds are calculated separately within permit series because different
# types of street work can have very different measurement distributions.
# Flags identify values for review; no records are removed or imputed.

measurement_checks = []

for column in [
    "permitlinearfeet",
    "permittotalsqfeet",
    "permitestimatednumberofcuts"
]:
    values = df_street[column]

    # Exclude negatives when estimating the non-negative measurement distribution.
    non_negative = values.where(values >= 0)

    grouped = non_negative.groupby(
        df_street["permit_series"],
        observed=True
    )

    q1 = grouped.transform("quantile", q=0.25)
    q3 = grouped.transform("quantile", q=0.75)
    observed_count = grouped.transform("count")

    upper_limit = q3 + 1.5 * (q3 - q1)

    negative_flag = values.lt(0).fillna(False)

    # Require enough observations to support a group-specific threshold.
    high_flag = (
        values.gt(upper_limit)
        & observed_count.ge(20)
    ).fillna(False)

    df_street[f"flag_negative_{column}"] = negative_flag
    df_street[f"flag_high_{column}"] = high_flag

    measurement_checks.append({
        "measurement": column,
        "missing": int(values.isna().sum()),
        "zero": int(values.eq(0).sum()),
        "negative": int(negative_flag.sum()),
        "high_IQR_within_series": int(high_flag.sum()),
        "maximum_observed": values.max()
    })

start = df_street["issuedworkstartdate"]
end = df_street["issuedworkenddate"]
issued = df_street["issued_date"]

df_street["flag_work_end_before_start"] = (
    end < start
).fillna(False)

# A start before issuance can occur for emergency/retroactive records.
# Flag it for interpretation rather than automatically calling it invalid.
df_street["flag_work_start_before_issue"] = (
    start.dt.normalize() < issued.dt.normalize()
).fillna(False)

df_street["flag_work_end_before_issue"] = (
    end.dt.normalize() < issued.dt.normalize()
).fillna(False)

measurement_summary = pd.DataFrame(measurement_checks)

date_summary = pd.Series({
    "Missing work start": start.isna().sum(),
    "Missing work end": end.isna().sum(),
    "Work end before work start":
        df_street["flag_work_end_before_start"].sum(),
    "Work start date before issuance date":
        df_street["flag_work_start_before_issue"].sum(),
    "Work end date before issuance date":
        df_street["flag_work_end_before_issue"].sum()
}, name="record_count").to_frame()

print("MEASUREMENT CHECKS")
display(measurement_summary)

print("\nWORK-DATE CHECKS")
display(date_summary)

print("\nNUMERIC DISTRIBUTIONS")
display(
    df_street[[
        "permitlinearfeet",
        "permittotalsqfeet",
        "permitestimatednumberofcuts"
    ]]
    .describe(percentiles=[0.5, 0.95, 0.99])
    .round(2)
)

measurement_summary.to_csv(
    STREET_OUTPUT_DIR / "street_measurement_checks.csv",
    index=False
)
date_summary.to_csv(
    STREET_OUTPUT_DIR / "street_work_date_checks.csv"
)

print(f"\nRows retained: {len(df_street):,}")

MEASUREMENT CHECKS


,measurement,missing,zero,negative,high_IQR_within_series,maximum_observed
0,permitlinearfeet,1169526,0,0,52906,9999
1,permittotalsqfeet,1756518,3209,0,37606,102595914
2,permitestimatednumberofcuts,1729665,5123,0,32669,9680



WORK-DATE CHECKS


,record_count
Missing work start,13749
Missing work end,13749
Work end before work start,0
Work start date before issuance date,44289
Work end date before issuance date,4208



NUMERIC DISTRIBUTIONS


,permitlinearfeet,permittotalsqfeet,permitestimatednumberofcuts
count,795703.0,208711.0,235564.0
mean,218.77,1702.45,4.36
std,585.0,224786.66,64.12
min,1.0,0.0,0.0
50%,75.0,100.0,1.0
95%,700.0,5000.0,10.0
99%,5000.0,10000.0,20.0
max,9999.0,102595914.0,9680.0



Rows retained: 1,965,229


In [10]:
# This cell defines which records enter our construction-permit analysis.
# We exclude non-construction administrative series and ambiguous review statuses,
# saving excluded records for audit. Expired permits, renewals and permits voided
# after issuance remain historical issuance activity.
# Overall and category summaries count distinct permit numbers.

excluded_series = {
    "MISCELLANEOUS CASH RECEIPTS",
    "VAULT LICENSE"
}

ambiguous_statuses = {
    "SUBMITTED",
    "FEE REVIEW",
    "PERMIT HELD FOR EXTERNAL REVIEW",
    "HIQA SUBMITTED"
}

series_exclusion = df_street["permit_series"].isin(excluded_series)
status_exclusion = df_street["permit_status"].isin(ambiguous_statuses)
exclude_mask = series_exclusion | status_exclusion

# Preserve excluded records and the reason for each exclusion.
excluded_street = df_street.loc[exclude_mask].copy()
excluded_street["excluded_series"] = series_exclusion.loc[exclude_mask]
excluded_street["excluded_status"] = status_exclusion.loc[exclude_mask]

excluded_street.to_csv(
    STREET_OUTPUT_DIR / "street_excluded_records.csv",
    index=False
)

street_cleaning_log = pd.DataFrame([
    {"step": "Downloaded records", "rows": len(df_street)},
    {
        "step": "Excluded series flag (may overlap)",
        "rows": int(series_exclusion.sum())
    },
    {
        "step": "Ambiguous status flag (may overlap)",
        "rows": int(status_exclusion.sum())
    },
    {
        "step": "Unique excluded records",
        "rows": int(exclude_mask.sum())
    },
    {
        "step": "Eligible records retained",
        "rows": int((~exclude_mask).sum())
    }
])

# Retain only eligible records in the working table.
df_street = df_street.loc[~exclude_mask].copy()

assert df_street["permitnumber"].notna().all()
assert not df_street["permitnumber"].duplicated().any()

df_street["year"] = df_street["issued_date"].dt.year
df_street["month"] = df_street["issued_date"].dt.month

monthly_street_counts = (
    df_street.groupby(
        ["borough", "year", "month"],
        observed=True,
        as_index=False
    )
    .agg(street_permit_count=("permitnumber", "nunique"))
    .sort_values(["borough", "year", "month"])
    .reset_index(drop=True)
)

monthly_street_by_category = (
    df_street.groupby(
        ["borough", "year", "month", "permit_category"],
        observed=True,
        as_index=False
    )
    .agg(street_permit_count=("permitnumber", "nunique"))
    .sort_values(["borough", "year", "month", "permit_category"])
    .reset_index(drop=True)
)

monthly_street_by_series = (
    df_street.groupby(
        ["borough", "year", "month", "permit_series"],
        observed=True,
        as_index=False
    )
    .agg(street_permit_count=("permitnumber", "nunique"))
    .sort_values(["borough", "year", "month", "permit_series"])
    .reset_index(drop=True)
)

expected_groups = pd.MultiIndex.from_product(
    [
        sorted(valid_boroughs),
        [2023, 2024, 2025],
        range(1, 13)
    ],
    names=["borough", "year", "month"]
)

observed_groups = pd.MultiIndex.from_frame(
    monthly_street_counts[["borough", "year", "month"]]
)
missing_groups = expected_groups.difference(observed_groups)

assert (
    monthly_street_counts["street_permit_count"].sum()
    == len(df_street)
)
assert not monthly_street_counts.duplicated(
    ["borough", "year", "month"]
).any()

# Save real-data outputs under new, explicit filenames.
for table, filename in [
    (monthly_street_counts, "street_monthly_counts.csv"),
    (monthly_street_by_category, "street_monthly_by_category.csv"),
    (monthly_street_by_series, "street_monthly_by_series.csv"),
    (street_cleaning_log, "street_cleaning_log.csv")
]:
    table.to_csv(STREET_OUTPUT_DIR / filename, index=False)

df_street.to_csv(
    STREET_OUTPUT_DIR / "street_permits_cleaned_2023_2025.csv",
    index=False,
    chunksize=50000
)

print("CLEANING LOG")
display(street_cleaning_log)

print(f"\nMonthly groups: {len(monthly_street_counts)} / 180")
print(f"Missing borough-month groups: {len(missing_groups)}")

if len(missing_groups):
    display(missing_groups.to_frame(index=False))

print("\nMONTHLY SUMMARY PREVIEW")
display(monthly_street_counts.head())

print("\nBOROUGH TOTALS")
display(
    monthly_street_counts.groupby("borough", observed=True)
    ["street_permit_count"]
    .sum()
    .to_frame()
)

print("\nCleaned data and monthly summaries saved.")

CLEANING LOG


,step,rows
0,Downloaded records,1965229
1,Excluded series flag (may overlap),13749
2,Ambiguous status flag (may overlap),109
3,Unique excluded records,13858
4,Eligible records retained,1951371



Monthly groups: 180 / 180
Missing borough-month groups: 0

MONTHLY SUMMARY PREVIEW


,borough,year,month,street_permit_count
0,BRONX,2023,1,5410
1,BRONX,2023,2,5584
2,BRONX,2023,3,8823
3,BRONX,2023,4,6649
4,BRONX,2023,5,6541



BOROUGH TOTALS


,street_permit_count
borough,
BRONX,263877
BROOKLYN,582064
MANHATTAN,519602
QUEENS,444200
STATEN ISLAND,141628



Cleaned data and monthly summaries saved.


In [ ]:
# Visualize real eligible street counts across boroughs as pre-integration EDA.
# This is additional EDA, not one of the assigned cross-domain research charts.
eda_street = monthly_street_counts.groupby('borough',observed=True,as_index=False)['street_permit_count'].sum()
fig_street_eda = px.bar(eda_street,x='borough',y='street_permit_count',
    title='Eligible street permits by borough, 2023–2025',template='plotly_white')
fig_street_eda.show()
fig_street_eda.write_html(str(STREET_OUTPUT_DIR/'eda_street_boroughs.html'),include_plotlyjs=True)

In [ ]:
# Load the actual integrated monthly dataset for cross-domain research.
task2_integrated = load_saved_csv('integrated_monthly_permits.csv')
assert len(task2_integrated) == 180
assert task2_integrated['_merge'].eq('both').all()
assert not task2_integrated.duplicated(['borough','year','month']).any()
assert task2_integrated['building_permit_count'].sum() == 408566
assert task2_integrated['street_permit_count'].sum() == 1951371

In [13]:
# This cell answers Q3: do building and street permits peak in the same months?
# It averages each calendar month's counts across the three complete years,
# then indexes each series to its borough mean (100 = average activity).
# Two heatmaps compare relative patterns; the peak table reports observed maxima.
# Three years describe this period but do not establish stable seasonality.

from plotly.subplots import make_subplots
import plotly.graph_objects as go

metrics = ["building_permit_count", "street_permit_count"]

# Require all 12 months in all 3 years for every borough.
coverage = (
    task2_integrated.groupby(["borough", "year"])["month"].nunique()
)
assert len(coverage) == 15 and coverage.eq(12).all()

seasonal = (
    task2_integrated.groupby(["borough", "month"], as_index=False)[metrics]
    .mean()
)

for column in metrics:
    borough_mean = seasonal.groupby("borough")[column].transform("mean")
    seasonal[f"{column}_index"] = (
        seasonal[column] / borough_mean.where(borough_mean > 0) * 100
    )

borough_order = sorted(seasonal["borough"].unique())
month_order = list(range(1, 13))
month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

fig_q3 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Building issuances", "Street permits"],
    shared_yaxes=True
)

for position, column in enumerate(metrics, start=1):
    matrix = seasonal.pivot(
        index="borough",
        columns="month",
        values=f"{column}_index"
    ).reindex(index=borough_order, columns=month_order)

    fig_q3.add_trace(
        go.Heatmap(
            z=matrix.to_numpy(),
            x=month_labels,
            y=borough_order,
            coloraxis="coloraxis",
            hovertemplate=(
                "Borough: %{y}<br>Month: %{x}<br>"
                "Activity index: %{z:.1f}<extra></extra>"
            )
        ),
        row=1,
        col=position
    )

fig_q3.update_layout(
    title="Q3: Average Calendar-Month Activity, 2023–2025",
    coloraxis={
        "colorscale": "RdBu_r",
        "cmid": 100,
        "colorbar": {"title": "Index"}
    },
    template="plotly_white",
    height=450,
    width=1100,
    margin={"l": 140}
)

fig_q3.show()

# Preserve ties rather than arbitrarily selecting one peak month.
peak_rows = []

for borough, group in seasonal.groupby("borough"):
    result = {"borough": borough}

    for column, label in [
        ("building_permit_count", "building_peak_months"),
        ("street_permit_count", "street_peak_months")
    ]:
        maximum = group[column].max()
        result[label] = ", ".join(
            month_labels[int(month) - 1]
            for month in group.loc[group[column].eq(maximum), "month"]
        )

    peak_rows.append(result)

q3_peaks = pd.DataFrame(peak_rows)

print("PEAK CALENDAR MONTHS — BASED ON THREE-YEAR AVERAGES")
display(q3_peaks)

fig_q3.write_html(
    str(STREET_OUTPUT_DIR / "q3_seasonal_comparison.html"),
    include_plotlyjs=True
)
seasonal.to_csv(
    STREET_OUTPUT_DIR / "q3_seasonal_data.csv", index=False
)
q3_peaks.to_csv(
    STREET_OUTPUT_DIR / "q3_peak_months.csv", index=False
)

print("Q3 chart and supporting tables saved.")

PEAK CALENDAR MONTHS — BASED ON THREE-YEAR AVERAGES


,borough,building_peak_months,street_peak_months
0,BRONX,Mar,Mar
1,BROOKLYN,May,Mar
2,MANHATTAN,May,Mar
3,QUEENS,Mar,Mar
4,STATEN ISLAND,Oct,Jun


Q3 chart and supporting tables saved.


In [14]:
# This cell compares building activity with the 10 largest street-permit categories.
# Categories are selected by total volume, not by the strength of their correlation.
# Correlations are calculated within each borough using all 36 matched months.
# An absent category in a covered month means zero recorded eligible permits.
# Undefined correlations remain missing; no artificial results are substituted.

category_keys = ["borough", "year", "month", "permit_category"]

category_data = monthly_street_by_category.copy()
category_data["borough"] = (
    category_data["borough"].astype("string").str.strip().str.upper()
)
category_data["permit_category"] = (
    category_data["permit_category"].astype("string")
)

assert not category_data.duplicated(category_keys).any()
assert category_data["permit_category"].notna().all()
assert (
    category_data["street_permit_count"].sum() == 1_951_371
)

category_totals = (
    category_data.groupby("permit_category", observed=True)
    ["street_permit_count"]
    .sum()
    .sort_values(ascending=False)
)

top_categories = category_totals.head(10).index.tolist()

base = task2_integrated[
    ["borough", "year", "month", "building_permit_count"]
].copy()

assert len(base) == 180
assert not base.duplicated(["borough", "year", "month"]).any()

# Create all covered borough-month/category combinations.
grid = base.merge(
    pd.DataFrame({"permit_category": top_categories}),
    how="cross"
)

comparison = grid.merge(
    category_data.loc[
        category_data["permit_category"].isin(top_categories)
    ],
    on=category_keys,
    how="left",
    validate="one_to_one"
)

comparison["street_permit_count"] = (
    comparison["street_permit_count"].fillna(0)
)

correlation_rows = []

for (borough, category), group in comparison.groupby(
    ["borough", "permit_category"], observed=True
):
    x = group["building_permit_count"].astype(float)
    y = group["street_permit_count"].astype(float)

    usable = x.notna() & y.notna()
    x, y = x[usable], y[usable]

    r = (
        x.corr(y)
        if len(x) >= 3 and x.nunique() > 1 and y.nunique() > 1
        else float("nan")
    )

    correlation_rows.append({
        "borough": borough,
        "permit_category": category,
        "matched_months": len(x),
        "months_with_category_activity": int(y.gt(0).sum()),
        "pearson_r": r
    })

q4_correlations = pd.DataFrame(correlation_rows)

matrix = q4_correlations.pivot(
    index="permit_category",
    columns="borough",
    values="pearson_r"
).reindex(top_categories)

fig_q4 = px.imshow(
    matrix,
    text_auto=".2f",
    aspect="auto",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    title="Q4: Building–Street Category Correlations Within Boroughs",
    labels={
        "x": "Borough",
        "y": "Street permit category",
        "color": "Pearson r"
    },
    template="plotly_white",
    height=750
)

fig_q4.show()

print("SELECTED CATEGORIES — TOTAL ELIGIBLE PERMITS")
display(category_totals.head(10).to_frame("permit_count"))

print("\nCORRELATIONS AND SAMPLE SIZES")
print(q4_correlations.round(4).to_string(index=False))

fig_q4.write_html(
    str(STREET_OUTPUT_DIR / "q4_category_correlations.html"),
    include_plotlyjs=True
)
q4_correlations.to_csv(
    STREET_OUTPUT_DIR / "q4_category_correlations.csv",
    index=False
)
comparison.to_csv(
    STREET_OUTPUT_DIR / "q4_category_monthly_comparison.csv",
    index=False
)

print("\nQ4 chart and supporting tables saved.")

SELECTED CATEGORIES — TOTAL ELIGIBLE PERMITS


,permit_count
permit_category,
PLACE EQUIPMENT OTHER THAN CRANE OR SHOV,331275
COMMERCIAL REFUSE CONTAINER,173250
OCCUPANCY OF ROADWAY AS STIPULATED,104544
OCCUPANCY OF SIDEWALK AS STIPULATED,101128
PLACE MATERIAL ON STREET,64541
MAJOR INSTALLATION - GAS,46820
UTILITY MANHOLE EMBARGO PERMIT,46703
REPAIR SIDEWALK,45362
PLACE OPEN CONSTRUCTION DEBRIS CONTAINER ON STREET,44722



CORRELATIONS AND SAMPLE SIZES
      borough                                    permit_category  matched_months  months_with_category_activity  pearson_r
        BRONX                        COMMERCIAL REFUSE CONTAINER              36                             36    -0.1831
        BRONX           COMMERCIAL REFUSE CONTAINER CRITICAL STS              36                             36    -0.2886
        BRONX                           MAJOR INSTALLATION - GAS              36                             36     0.4421
        BRONX                 OCCUPANCY OF ROADWAY AS STIPULATED              36                             36     0.0208
        BRONX                OCCUPANCY OF SIDEWALK AS STIPULATED              36                             36     0.0102
        BRONX           PLACE EQUIPMENT OTHER THAN CRANE OR SHOV              36                             36     0.1987
        BRONX                           PLACE MATERIAL ON STREET              36                            

### Street dataset: methodology and findings

#### Source and extraction

Dataset: Street Construction Permits (2022–Present)
https://data.cityofnewyork.us/d/tqtj-sjs8

The source contains 39 columns. We retained 20 genuine source fields
and downloaded 1,965,229 records with permit issuance dates during
2023–2025. Filtering used permitissuedate, not record creation date.

Records were downloaded in batches ordered by the API row identifier.
Interrupted requests were retried, and downloading resumed from the
saved CSV. The final count matched the initial API count.

The dataset is live: count agreement does not establish a frozen
snapshot. Subsequent checks found no repeated permit numbers.

The earlier draft's synthetic fallback and results were discarded.
All final analysis uses real downloaded records.

#### Cleaning and counting

Text spacing and borough labels were standardized. Dates and numeric
measurements were converted successfully, with no conversion failures.
Identifiers were preserved as text.

There were no missing permit numbers, repeated permit numbers,
unexpected boroughs or out-of-scope issuance dates.

We excluded 13,749 records belonging to miscellaneous cash receipts
or vault licences and 109 records with ambiguous review/submission
statuses. These are analytical scope decisions.

This retained 1,951,371 eligible records. Each unique permit number
is counted once. Renewals, reissues and extensions remain activity;
counts therefore do not represent unique construction projects.

Expired permits and records voided after issue remain historical
issuance activity. These counts must not be interpreted as currently
valid authorizations.

Missing work measurements and previous/next permit links were retained
without zero imputation. Their absence does not prevent borough/month
analysis.

No negative measurements were found. High values were flagged using
upper IQR thresholds within permit series and retained for review.
They were not assumed to be errors or used to weight permit counts.

There were 44,289 records with work starting before issuance and
4,208 with work ending before issuance. These were flagged and retained;
their causes were not verified. No records had work end before start.

#### Aggregation and validation

Overall counts were aggregated by borough, year and month.
Separate summaries were produced by detailed permit category and
broader permit series.

All 180 expected borough-month groups contained eligible records.
Overall monthly counts summed to 1,951,371, with unique join keys.

#### Question 3: Do the busiest calendar months coincide?

We averaged monthly counts across 2023–2025 for each calendar month.
Each domain was indexed to its own borough average, where 100 means
average activity.

| Borough | Building peak | Street peak |
|---|---|---|
| Bronx | March | March |
| Brooklyn | May | March |
| Manhattan | May | March |
| Queens | March | March |
| Staten Island | October | June |

Peak calendar months coincide in the Bronx and Queens. The other
three boroughs have different peak months.

These are peaks in three-year calendar-month averages, not individual
record-high months. Three years do not establish stable seasonality.
Peak agreement alone does not establish coordination between domains.

#### Question 4: Does the relationship vary by street category?

We selected the ten largest detailed categories by total eligible
permit volume, before examining their correlations.

Pearson correlations were calculated separately within each borough,
using 36 matched months per category. Categories absent from a covered
month were assigned zero recorded eligible permits.

The strongest observed positive correlation among the selected
comparisons was Staten Island's COMMERCIAL REFUSE CONTAINER CRITICAL
STS category (r = 0.6105). The strongest negative correlation was
Staten Island's REPAIR SIDEWALK category (r = -0.6226).

Other examples include Manhattan's OCCUPANCY OF SIDEWALK AS STIPULATED
(r = 0.4540), Bronx's MAJOR INSTALLATION - GAS (r = 0.4421), and
Brooklyn's PLACE MATERIAL ON STREET (r = 0.4154).

These differences show that overall street-permit totals can conceal
distinct category patterns. Staten Island's near-zero overall
correlation does not mean every category has a near-zero relationship.

#### Limitations and potential use

These are exploratory descriptive comparisons across 50 borough/category
combinations. No significance testing or multiple-comparison adjustment
was performed. Shared trends, seasonality and serial dependence may
influence correlations.

The chart covers the ten highest-volume categories, not every category.
Some categories have months without recorded activity; activity-month
counts are provided in the supporting table.

Permits are compared geographically and temporally, not linked to the
same project. The results do not establish causation, completed work,
congestion or infrastructure adequacy.

The findings can guide more detailed investigation of specific
borough/category combinations rather than support automatic policy
decisions.

### AI assistance and prompt record

ChatGPT assisted with source review, Python implementation, interpretation of diagnostic outputs and notebook organization. Real outputs were reviewed before reporting conclusions. Prompts included:

- “analyse the 3 things with the project description along with the task list” — review of the three contributions and missing requirements.
- “is there anything in the meantime to do in the other 2 tasks?” — preparation of integration and research-chart functions.
- “after every code block can you send me a small comment to add at the top?” — explanatory code comments.

Follow-up messages supplied executed outputs for diagnosis and interpretation. During the final review, ChatGPT reordered cells, replaced repeated live downloads with saved-snapshot loading, removed obsolete code and added validation. The notebook code records the implementation; reported results are grounded in real data rather than generated example values.
